In [4]:
import os
import sys
sys.path.append("kaggle/input/polymer_pipeline")

In [23]:
from data_preparation import get_data_paths, load_and_split_data, smiles_to_data
import model

In [7]:
os.environ['NEURIPS_DATA_PATH']     = 'kaggle/input/neurips-open-polymer-prediction-2025'
os.environ['EXTRA_DATA_BASE']       = 'kaggle/input/smiles-extra-data'
os.environ['TC_DATA_BASE']          = 'kaggle/input/tc-smiles'

In [8]:
paths = get_data_paths()
for k, v in paths.items():
    print(f"{k}: {v}")

train_csv: kaggle/input/neurips-open-polymer-prediction-2025/train.csv
test_csv: kaggle/input/neurips-open-polymer-prediction-2025/test.csv
sample_submission: kaggle/input/neurips-open-polymer-prediction-2025/sample_submission.csv
tc_data: kaggle/input/tc-smiles/Tc_SMILES.csv
tg_jcim_data: kaggle/input/smiles-extra-data/JCIM_sup_bigsmiles.csv
tg_excel_data: kaggle/input/smiles-extra-data/data_tg3.xlsx
density_data: kaggle/input/smiles-extra-data/data_dnst1.xlsx
supplement_dir: kaggle/input/neurips-open-polymer-prediction-2025/train_supplement
ffv_data: kaggle/input/neurips-open-polymer-prediction-2025/train_supplement/dataset4.csv
dataset1: kaggle/input/neurips-open-polymer-prediction-2025/train_supplement/dataset1.csv
dataset2: kaggle/input/neurips-open-polymer-prediction-2025/train_supplement/dataset2.csv
dataset3: kaggle/input/neurips-open-polymer-prediction-2025/train_supplement/dataset3.csv


In [5]:
train_df, val_df, test_df = load_and_split_data(paths)
print("Loaded:", len(train_df), len(val_df), len(test_df))

原始训练: 7973 条
  → 正在增强 Tc 数据，共 874 条
cross_smiles: 737 | 填充: 0
新增样本: 129 条
  → 正在增强 Tg 数据，共 662 条
cross_smiles: 526 | 填充: 15
新增样本: 136 条
  → 正在增强 Tg 数据，共 501 条
cross_smiles: 0 | 填充: 0
新增样本: 499 条
  → 正在增强 Density 数据，共 787 条


[11:44:30] SMILES Parse Error: syntax error while parsing: *O[Si](*)([R])[R]
[11:44:30] SMILES Parse Error: Failed parsing SMILES '*O[Si](*)([R])[R]' for input: '*O[Si](*)([R])[R]'
[11:44:30] SMILES Parse Error: syntax error while parsing: *NC(=O)c4ccc3c(=O)n(c2ccc([R]c1ccc(*)cc1)cc2)c(=O)c3c4
[11:44:30] SMILES Parse Error: Failed parsing SMILES '*NC(=O)c4ccc3c(=O)n(c2ccc([R]c1ccc(*)cc1)cc2)c(=O)c3c4' for input: '*NC(=O)c4ccc3c(=O)n(c2ccc([R]c1ccc(*)cc1)cc2)c(=O)c3c4'
[11:44:30] SMILES Parse Error: syntax error while parsing: O=C=N[R1]N=C=O.O[R2]O.O[R3]O
[11:44:30] SMILES Parse Error: Failed parsing SMILES 'O=C=N[R1]N=C=O.O[R2]O.O[R3]O' for input: 'O=C=N[R1]N=C=O.O[R2]O.O[R3]O'
[11:44:30] SMILES Parse Error: syntax error while parsing: *CN([R'])Cc2cc([R]c1cc(*)c(O)c(CN([R'])C*)c1)cc(*)c2O
[11:44:30] SMILES Parse Error: Failed parsing SMILES '*CN([R'])Cc2cc([R]c1cc(*)c(O)c(CN([R'])C*)c1)cc(*)c2O' for input: '*CN([R'])Cc2cc([R]c1cc(*)c(O)c(CN([R'])C*)c1)cc(*)c2O'
[11:44:30] SMILES Parse 

cross_smiles: 254 | 填充: 110
新增样本: 525 条
  → 正在增强 FFV 数据，共 862 条
cross_smiles: 43 | 填充: 43
新增样本: 819 条
Loaded: 8064 1008 1009


In [6]:
from train_stage3 import prepare_property_datasets, finetune_property

In [7]:
properties = ["Tg", "FFV", "Tc", "Density", "Rg"]
datasets = prepare_property_datasets(properties, train_df, val_df, test_df)

In [9]:
tg_data = datasets["Tg"]
train_tg, val_tg, test_tg = tg_data["train"], tg_data["val"], tg_data["test"]
finetune_property(
    train_tg,
    val_tg,
    property_name="Tg",
    best_params_path="stage2_final_model/final_stage2_params.pt",
    stage2_encoder_path="stage2_final_model/final_stage2_encoder.pt",
    stage2_predictor_path="stage2_final_model/final_stage2_predictor.pt",
    output_dir="stage3_heads",
    num_epochs=5000,
    batch_size=16,
    patience=20
)

📦 构建 PolymerDataset，样本数=924
   成功转换为图数据: 924 条
📦 构建 PolymerDataset，样本数=111
   成功转换为图数据: 111 条
[Tg] Epoch 001 Train MSE=13408.2056, MAE=92.7449, Val   MSE=11307.9082, MAE=80.8112
[Tg] Epoch 002 Train MSE=12410.8373, MAE=85.6316, Val   MSE=10765.5130, MAE=77.5131
[Tg] Epoch 003 Train MSE=12474.8950, MAE=84.6880, Val   MSE=10393.6205, MAE=75.9926
[Tg] Epoch 004 Train MSE=12325.6717, MAE=83.9503, Val   MSE=10180.4624, MAE=74.4176
[Tg] Epoch 005 Train MSE=12377.5903, MAE=83.2414, Val   MSE=10030.7661, MAE=73.9075
[Tg] Epoch 006 Train MSE=12264.0559, MAE=82.8960, Val   MSE=9931.9432, MAE=73.5641
[Tg] Epoch 007 Train MSE=12147.6778, MAE=82.6781, Val   MSE=9876.1585, MAE=72.9867
[Tg] Epoch 008 Train MSE=12076.6339, MAE=82.3133, Val   MSE=9778.9421, MAE=72.8230
[Tg] Epoch 009 Train MSE=11915.4748, MAE=81.9411, Val   MSE=9697.9814, MAE=72.5534
[Tg] Epoch 010 Train MSE=11873.5098, MAE=81.6726, Val   MSE=9636.7744, MAE=72.0139
[Tg] Epoch 011 Train MSE=11784.1381, MAE=81.2252, Val   MSE=9538.1887, 

In [11]:
ffv_data = datasets["FFV"]
train_ffv, val_ffv, test_ffv = ffv_data["train"], ffv_data["val"], ffv_data["test"]
finetune_property(
    train_ffv,
    val_ffv,
    property_name="FFV",
    best_params_path="stage2_final_model/final_stage2_params.pt",
    stage2_encoder_path="stage2_final_model/final_stage2_encoder.pt",
    stage2_predictor_path="stage2_final_model/final_stage2_predictor.pt",
    output_dir="stage3_heads",
    num_epochs=5000,
    batch_size=16,
    patience=20
)

📦 构建 PolymerDataset，样本数=6320
   成功转换为图数据: 6320 条
📦 构建 PolymerDataset，样本数=794
   成功转换为图数据: 794 条
[FFV] Epoch 001 Train MSE=555.6694, MAE=10.4090, Val   MSE=0.0030, MAE=0.0383
[FFV] Epoch 002 Train MSE=0.0022, MAE=0.0350, Val   MSE=0.0015, MAE=0.0295
[FFV] Epoch 003 Train MSE=0.0018, MAE=0.0261, Val   MSE=0.0009, MAE=0.0243
[FFV] Epoch 004 Train MSE=0.0010, MAE=0.0219, Val   MSE=0.0008, MAE=0.0200
[FFV] Epoch 005 Train MSE=0.0012, MAE=0.0216, Val   MSE=0.0007, MAE=0.0203
[FFV] Epoch 006 Train MSE=0.0009, MAE=0.0212, Val   MSE=0.0007, MAE=0.0193
[FFV] Epoch 007 Train MSE=0.0026, MAE=0.0215, Val   MSE=0.0007, MAE=0.0195
[FFV] Epoch 008 Train MSE=0.0010, MAE=0.0216, Val   MSE=0.0010, MAE=0.0230
[FFV] Epoch 009 Train MSE=0.0009, MAE=0.0208, Val   MSE=0.0007, MAE=0.0189
[FFV] Epoch 010 Train MSE=0.0009, MAE=0.0208, Val   MSE=0.0009, MAE=0.0207
[FFV] Epoch 011 Train MSE=0.0011, MAE=0.0213, Val   MSE=0.0007, MAE=0.0189
[FFV] Epoch 012 Train MSE=0.0009, MAE=0.0209, Val   MSE=0.0007, MAE=0.0188
[

In [12]:
Tc_data = datasets["Tc"]
train_Tc, val_Tc, test_Tc = Tc_data["train"], Tc_data["val"], Tc_data["test"]
finetune_property(
    train_Tc,
    val_Tc,
    property_name="Tc",
    best_params_path="stage2_final_model/final_stage2_params.pt",
    stage2_encoder_path="stage2_final_model/final_stage2_encoder.pt",
    stage2_predictor_path="stage2_final_model/final_stage2_predictor.pt",
    output_dir="stage3_heads",
    num_epochs=5000,
    batch_size=16,
    patience=20
)

📦 构建 PolymerDataset，样本数=707
   成功转换为图数据: 707 条
📦 构建 PolymerDataset，样本数=81
   成功转换为图数据: 81 条
[Tc] Epoch 001 Train MSE=2985.4538, MAE=46.9410, Val   MSE=2015.5611, MAE=37.5837
[Tc] Epoch 002 Train MSE=809.9915, MAE=21.3029, Val   MSE=545.6427, MAE=15.7766
[Tc] Epoch 003 Train MSE=281.5529, MAE=11.4636, Val   MSE=241.5754, MAE=10.3295
[Tc] Epoch 004 Train MSE=112.8312, MAE=6.9962, Val   MSE=61.3945, MAE=5.4252
[Tc] Epoch 005 Train MSE=19.5073, MAE=2.8387, Val   MSE=2.4153, MAE=1.1042
[Tc] Epoch 006 Train MSE=0.8686, MAE=0.5560, Val   MSE=0.6365, MAE=0.4846
[Tc] Epoch 007 Train MSE=0.4033, MAE=0.3208, Val   MSE=0.2657, MAE=0.3295
[Tc] Epoch 008 Train MSE=0.1154, MAE=0.2236, Val   MSE=0.1548, MAE=0.2583
[Tc] Epoch 009 Train MSE=0.1067, MAE=0.1988, Val   MSE=0.1071, MAE=0.2229
[Tc] Epoch 010 Train MSE=0.0513, MAE=0.1642, Val   MSE=0.0942, MAE=0.2091
[Tc] Epoch 011 Train MSE=0.0507, MAE=0.1598, Val   MSE=0.0774, MAE=0.1928
[Tc] Epoch 012 Train MSE=0.0412, MAE=0.1483, Val   MSE=0.0795, MAE=0.1

In [13]:
Density_data = datasets["Density"]
train_Density, val_Density, test_Density = Density_data["train"], Density_data["val"], Density_data["test"]
finetune_property(
    train_Density,
    val_Density,
    property_name="Density",
    best_params_path="stage2_final_model/final_stage2_params.pt",
    stage2_encoder_path="stage2_final_model/final_stage2_encoder.pt",
    stage2_predictor_path="stage2_final_model/final_stage2_predictor.pt",
    output_dir="stage3_heads",
    num_epochs=5000,
    batch_size=16,
    patience=20
)

📦 构建 PolymerDataset，样本数=1025
   成功转换为图数据: 1025 条
📦 构建 PolymerDataset，样本数=112
   成功转换为图数据: 112 条
[Density] Epoch 001 Train MSE=7501.5719, MAE=70.3908, Val   MSE=2393.2037, MAE=36.9903
[Density] Epoch 002 Train MSE=1290.5377, MAE=23.4276, Val   MSE=625.1277, MAE=17.3421
[Density] Epoch 003 Train MSE=307.1125, MAE=11.0893, Val   MSE=50.3685, MAE=5.1496
[Density] Epoch 004 Train MSE=15.6414, MAE=2.2201, Val   MSE=4.6160, MAE=1.2598
[Density] Epoch 005 Train MSE=4.0515, MAE=1.0938, Val   MSE=3.2986, MAE=1.0719
[Density] Epoch 006 Train MSE=2.1617, MAE=0.9213, Val   MSE=2.5001, MAE=0.9812
[Density] Epoch 007 Train MSE=1.9937, MAE=0.8204, Val   MSE=1.7329, MAE=0.8232
[Density] Epoch 008 Train MSE=1.0424, MAE=0.6591, Val   MSE=1.3190, MAE=0.7355
[Density] Epoch 009 Train MSE=0.8940, MAE=0.6194, Val   MSE=1.1276, MAE=0.6968
[Density] Epoch 010 Train MSE=0.7547, MAE=0.5854, Val   MSE=0.9283, MAE=0.6437
[Density] Epoch 011 Train MSE=0.6965, MAE=0.5663, Val   MSE=0.8205, MAE=0.6189
[Density] Epoch

In [14]:
Rg_data = datasets["Rg"]
train_Rg, val_Rg, test_Rg = Rg_data["train"], Rg_data["val"], Rg_data["test"]
finetune_property(
    train_Rg,
    val_Rg,
    property_name="Rg",
    best_params_path="stage2_final_model/final_stage2_params.pt",
    stage2_encoder_path="stage2_final_model/final_stage2_encoder.pt",
    stage2_predictor_path="stage2_final_model/final_stage2_predictor.pt",
    output_dir="stage3_heads",
    num_epochs=5000,
    batch_size=16,
    patience=20
)

📦 构建 PolymerDataset，样本数=519
   成功转换为图数据: 519 条
📦 构建 PolymerDataset，样本数=49
   成功转换为图数据: 49 条
[Rg] Epoch 001 Train MSE=5206.0480, MAE=65.0469, Val   MSE=4075.7390, MAE=57.5508
[Rg] Epoch 002 Train MSE=2700.0602, MAE=46.5232, Val   MSE=1833.9158, MAE=37.4224
[Rg] Epoch 003 Train MSE=1123.7114, MAE=27.4333, Val   MSE=1037.5559, MAE=24.7466
[Rg] Epoch 004 Train MSE=726.5692, MAE=21.0905, Val   MSE=698.3133, MAE=20.4873
[Rg] Epoch 005 Train MSE=524.9166, MAE=17.9167, Val   MSE=477.8667, MAE=17.3084
[Rg] Epoch 006 Train MSE=346.6278, MAE=15.2477, Val   MSE=296.5696, MAE=14.0274
[Rg] Epoch 007 Train MSE=185.5356, MAE=11.2682, Val   MSE=162.2657, MAE=9.9594
[Rg] Epoch 008 Train MSE=101.8267, MAE=7.8690, Val   MSE=103.7641, MAE=7.2786
[Rg] Epoch 009 Train MSE=71.3106, MAE=6.2899, Val   MSE=98.7221, MAE=6.9873
[Rg] Epoch 010 Train MSE=72.2925, MAE=6.1995, Val   MSE=89.3450, MAE=6.8310
[Rg] Epoch 011 Train MSE=68.2705, MAE=6.1629, Val   MSE=89.8648, MAE=6.7089
[Rg] Epoch 012 Train MSE=63.4457, MAE

In [29]:
import torch
import pandas as pd
import torch.nn as nn
# 五个属性
PROPERTIES = ["Tg", "FFV", "Tc", "Density", "Rg"]

# 全局超参和设备
BEST_PARAMS = torch.load("stage2_final_model/final_stage2_params.pt")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Stage 3 微调后模型路径
STAGE3_DIR = "stage3_heads"

# 模型缓存
_model_cache = {}

In [32]:
from model import WDMPNN, GraphPredictor

def predict_for_smile(smiles: str) -> dict:
 
    """
    对一个 SMILES 字符串预测所有属性，返回 {property: value}
    """
    if not _model_cache:
        for prop in PROPERTIES:
            # 加载权重
            enc = WDMPNN(9, 4, BEST_PARAMS["hidden_dim"], BEST_PARAMS["num_edge_layers"]).to(DEVICE)
            enc.load_state_dict(torch.load(f"{STAGE3_DIR}/encoder_ft_{prop}.pt", map_location=DEVICE))
            enc.eval()

            pred = GraphPredictor(BEST_PARAMS["hidden_dim"], [BEST_PARAMS["hidden_dim"] // 2], 1).to(DEVICE)
            pred.load_state_dict(torch.load(f"{STAGE3_DIR}/predictor_ft_{prop}.pt", map_location=DEVICE))
            pred.eval()

            down = nn.Sequential(
                nn.Linear(1, 32),
                nn.ReLU(),
                nn.Linear(32, 1)
            ).to(DEVICE)
            down.load_state_dict(torch.load(f"{STAGE3_DIR}/downstream_{prop}.pt", map_location=DEVICE))
            down.eval()

            _model_cache[prop] = (enc, pred, down)

    # 预处理单个分子
    data = smiles_to_data(smiles)
    data = data.to(DEVICE)
    batch = torch.zeros(data.x.shape[0], dtype=torch.long).to(DEVICE)

    results = {}
    for prop in PROPERTIES:
        encoder, predictor, downstream = _model_cache[prop]
        with torch.no_grad():
            h = encoder(data.x, data.edge_index, data.edge_attr,
                        torch.ones(data.edge_attr.size(0), device=DEVICE), batch)
            h = predictor(h).view(-1, 1)
            out = downstream(h)
            results[prop] = out.item()

    return results

In [33]:
# 读取 test.csv
df_test = pd.read_csv(paths['test_csv'], dtype={"id": str})

# 批量预测
out_records = []
for _, row in df_test.iterrows():
    _id, smi = row["id"], row["SMILES"]
    try:
        preds = predict_for_smile(smi)
    except Exception:
        preds = {p: float("nan") for p in PROPERTIES}
    rec = {"id": _id}
    rec.update(preds)
    out_records.append(rec)

In [34]:
df_out = pd.DataFrame(out_records, columns=["id"] + PROPERTIES)
df_out.to_csv("submission.csv", index=False)
print("✅ Saved submission.csv with", len(df_out), "rows.")

# 可选预览
df_out.head()

✅ Saved submission.csv with 3 rows.


,id,Tg,FFV,Tc,Density,Rg
0,1109053969,116.510162,0.371470,0.213109,1.219039,22.131180
1,1422188626,148.291412,0.388503,0.221118,1.023798,19.461994
2,2032016830,190.065063,0.357709,0.258205,1.039335,22.102510
